In [1]:
# ============================================
# TRAIN MODELS WITH FEATURE ENGINEERING (80/20)
# ============================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import recall_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
import joblib
import os

# -----------------------------
# Load real dataset
# -----------------------------
df = pd.read_csv(r"C:\Users\shiro\OneDrive\Desktop\Python files\Maternal_health_risk_Project\VIz Project\data\synthetic_maternal_cleaned.csv")

# -----------------------------
# FEATURE ENGINEERING
# -----------------------------
df["MAP"] = (df["SystolicBP"] + 2 * df["DiastolicBP"]) / 3
df["PulsePressure"] = df["SystolicBP"] - df["DiastolicBP"]
df["ShockIndex"] = df["HeartRate"] / df["SystolicBP"]
df["BPRatio"] = df["SystolicBP"] / df["DiastolicBP"]

df["TempDeviation"] = df["BodyTemp"] - 98.2
df["BSDeviation"] = df["BS"] - 7

df["CombinedRiskScore"] = (
    (df["MAP"] > 105).astype(int) +
    (df["BS"] > 10).astype(int) +
    (df["HeartRate"] > 90).astype(int) +
    (df["TempDeviation"] > 1).astype(int)
)

# -----------------------------
# Split X and y
# -----------------------------
X = df.drop("RiskLevel", axis=1)
y = df["RiskLevel"]

# -----------------------------
# Encode labels
# -----------------------------
le = LabelEncoder()
y_enc = le.fit_transform(y)

# -----------------------------
# 80/20 Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

# -----------------------------
# Define models
# -----------------------------
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(max_iter=5000))
    ]),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier(
        objective="multi:softprob",
        num_class=len(np.unique(y_enc)),
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42
    )
}

# Voting & Stacking Ensembles
estimators = [
    ("lr", LogisticRegression(max_iter=5000)),
    ("dt", DecisionTreeClassifier()),
    ("rf", RandomForestClassifier()),
    ("gb", GradientBoostingClassifier())
]

models["Voting Ensemble"] = VotingClassifier(
    estimators=estimators,
    voting="soft"
)

models["Stacking Ensemble"] = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=5000)
)

# -----------------------------
# Train + Evaluate + Save Models
# -----------------------------
results = {}

os.makedirs("../models_FE", exist_ok=True)

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    macro_recall = recall_score(y_test, preds, average="macro")
    results[name] = macro_recall
    
    print(f"Macro Recall: {macro_recall:.4f}")
    print(classification_report(y_test, preds))
    
    # Save model
    joblib.dump(model, f"../models_FE/{name.replace(' ', '_')}.pkl")

# -----------------------------
# Comparison Table
# -----------------------------
comparison_df = pd.DataFrame.from_dict(results, orient="index", columns=["Macro Recall"])
comparison_df = comparison_df.sort_values(by="Macro Recall", ascending=False)

comparison_df



Training Logistic Regression...
Macro Recall: 0.5898
              precision    recall  f1-score   support

           0       0.50      0.20      0.29         5
           1       0.87      0.96      0.91       227
           2       0.83      0.60      0.70        86

    accuracy                           0.86       318
   macro avg       0.73      0.59      0.63       318
weighted avg       0.85      0.86      0.84       318


Training Decision Tree...
Macro Recall: 0.8414
              precision    recall  f1-score   support

           0       0.50      0.60      0.55         5
           1       0.99      0.98      0.98       227
           2       0.94      0.94      0.94        86

    accuracy                           0.97       318
   macro avg       0.81      0.84      0.82       318
weighted avg       0.97      0.97      0.97       318


Training Random Forest...
Macro Recall: 0.8497
              precision    recall  f1-score   support

           0       1.00      0.60

,Macro Recall
XGBoost,0.920237
Gradient Boosting,0.857446
Voting Ensemble,0.855978
Stacking Ensemble,0.855978
Random Forest,0.849694
Decision Tree,0.841413
Logistic Regression,0.589803
